In [ ]:
import os
os.environ['KMP_DUPLICATE_LIB_OK']='True'

In [ ]:
import json
import os.path
import torch
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import pickle
from shapely.geometry import Point
import scienceplots
plt.style.use(['science', "no-latex"])
import random
import time

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)


In [ ]:
# import os
# os.environ['CUDA_LAUNCH_BLOCKING'] = "1"

In [ ]:
kitgreen="#009682"
green="#98BF64"
kitblue="#4664AA"
grey5="#f2f2f2ff"
grey30="#b3b3b3ff"

In [ ]:
# seed = random.randint(0, 10000)
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [ ]:
train_interested_locations = {
    "TLH2": {},
    "TLH3": {},
    "TLJ1":{},
    "TLF1":{},
    "TLF2":{},
    "TLC1":{},
    "TLC2":{},
    "TLD6":{},
    "TLG1":{},
    "TLG2":{},
    "TLE4":{},
}

In [ ]:
test_interested_locations = {
    "London": {},
    "TLH1": {},
    "TLE3":{},
    "TLD3":{},
    "TLD4":{},
}

In [ ]:
from torch_geometric.loader import DataLoader
from torch_geometric.data import Data, Batch

In [ ]:
train_graph_data_list = []

# Assuming you have multiple graph data, each in the following format:
for i, location in enumerate(train_interested_locations):  # your_multiple_graphs is your multi-graph data
    with open(f"./results/intermediate/GNN/{location}_prepared_graph_data.pt", "rb") as f:
        graph_data = torch.load(f, weights_only=False)

    train_graph_data_list.append(graph_data)

train_dataloader = DataLoader(train_graph_data_list, batch_size=1, shuffle=False)

In [ ]:
test_graph_data_list = []

# Assuming you have multiple graph data, each in the following format:
for i, location in enumerate(test_interested_locations):  # your_multiple_graphs is your multi-graph data
    with open(f"./results/intermediate/GNN/{location}_prepared_graph_data.pt", "rb") as f:
        graph_data = torch.load(f, weights_only=False)

    test_graph_data_list.append(graph_data)

test_dataloader = DataLoader(test_graph_data_list, batch_size=1, shuffle=False)

In [ ]:
from SpatialAllocation.GNN.core.EdgeWeightSolver import EdgeWeightSolver
from SpatialAllocation.GNN.core.ModelConfig import ModelConfig

In [ ]:
epochs = 1000
config = ModelConfig(
        hidden_dim=256,
        embedding_dim=128,
        num_layers=3,
        conv_type="hgt",  # Options: 'gcn', 'sage', 'gat', 'gin', 'hgt'
        allocation_temperature_start=0.01,
        clip_grad_norm=None,
        learning_rate=1e-3,
        weight_decay=1e-4,
        epochs=epochs,
        device='cuda',
        debug=False,
        use_scheduler = True,
        cosine_epochs = None,  # Cycle length; if None, uses total number of epochs
        cosine_eta_min = 5e-5,  # Minimum learning rate
        save_path=f"results/intermediate/gnn_model_SA_{time.time()}_London.pth",
        learnable=False,
    )

solver = EdgeWeightSolver(config)

In [ ]:
weights = {
    # 'supervised_substation_demand_loss': 1.0,
    # 'entropy_regularization': 0.05,
    # 'feature_similarity_loss': 1,
    # 'feature_consistency_loss': 1,
    'landuse_prediction_loss': 1.0,
}
# solver.train_multi_graph(train_dataloader, test_dataloader, weights)
solver.train_multi_graph(train_dataloader, objective_weights=weights)

In [ ]:
interested_locations = list(train_interested_locations.keys()) + list(test_interested_locations.keys())

In [ ]:
graph_data_list = []

# Assuming you have multiple graph data, each in the following format:
for i, location in enumerate(interested_locations):  # your_multiple_graphs is your multi-graph data
    with open(f"./results/intermediate/GNN/{location}_prepared_graph_data.pt", "rb") as f:
        graph_data = torch.load(f, weights_only=False)

    graph_data_list.append(graph_data)

dataloader = DataLoader(graph_data_list, batch_size=1, shuffle=True)

In [ ]:
with open("./results/intermediate/static_matrix.pickle", "rb") as f:
    matrix = pickle.load(f)

In [ ]:
cols = interested_locations
corr_matrix = matrix["corr_matrix"]
rmse_matrix = matrix["rmse_matrix"]
mae_matrix = matrix["mae_matrix"]

In [ ]:
ITL3_region = gpd.read_file(r"./results/intermediate/ITL3_region.gpkg")
substations = gpd.read_file(r"./results/intermediate/substations.gpkg")

In [ ]:
for i, location in enumerate(interested_locations):
    print(location)
    interested_region = ITL3_region[ITL3_region['ITL2'] == location].reset_index()
    interested_substations = substations[substations['ITL2'] == location].reset_index()

    with open(f"./results/intermediate/GridPoint/{location}_grid_points.pickle", "rb") as f:
        grid_gdf, step_size_m = pickle.load(f)
    grid_gdf = grid_gdf.reset_index()

    voronoi_gdf = gpd.read_file(f"./results/intermediate/voronoi/voronoi_{location}_without_subcolumn.gpkg")

    edge_weights = solver.predict_edge_weights(graph_data_list[i])
    edge_weights = edge_weights.set_index("agent_node_idx").sort_index()["predicted_weight"].values
    grid_gdf["GNN"] = edge_weights * grid_gdf["Demand (MVA)"]

    def calculate_demand(polygon):
        # Find all points located within this polygon
        points_within = grid_gdf[grid_gdf.geometry.within(polygon.geometry)]

        # If there are points within the polygon, calculate the sum of their demand; otherwise return 0
        if not points_within.empty:
            return points_within['GNN'].sum()
        else:
            return 0

    voronoi_gdf['demand'] = voronoi_gdf.apply(calculate_demand, axis=1)

    corr_matrix.loc["GNN", location] = interested_substations["Demand (MVA)"].corr(voronoi_gdf['demand'])
    rmse_matrix.loc["GNN", location] = np.sqrt(((interested_substations["Demand (MVA)"] - voronoi_gdf['demand']) ** 2).mean())
    mae_matrix.loc["GNN", location] = np.abs(interested_substations["Demand (MVA)"] - voronoi_gdf['demand']).mean()

In [ ]:
for i, location in enumerate(interested_locations):
    print(location)
    interested_region = ITL3_region[ITL3_region['ITL2'] == location].reset_index()
    interested_substations = substations[substations['ITL2'] == location].reset_index()

    with open(f"./results/intermediate/GridPoint/{location}_grid_points.pickle", "rb") as f:
        grid_gdf, step_size_m = pickle.load(f)
    grid_gdf = grid_gdf.reset_index()

    with open(f"./results/intermediate/Pyomo/hdbscan_voronoi_result_british_{location}.pkl", "rb") as f:
        voronoi_gdf = pickle.load(f)

    edge_weights = solver.predict_edge_weights(graph_data_list[i])
    edge_weights = edge_weights.set_index("agent_node_idx").sort_index()["predicted_weight"].values
    grid_gdf["GNN"] = edge_weights * grid_gdf["Demand (MVA)"]

    def calculate_demand(polygon):
        # Find all points located within this polygon
        points_within = grid_gdf[grid_gdf.geometry.within(polygon.geometry)]

        # If there are points within the polygon, calculate the sum of their demand; otherwise return 0
        if not points_within.empty:
            return points_within['GNN'].sum()
        else:
            return 0

    voronoi_gdf['demand'] = voronoi_gdf.apply(calculate_demand, axis=1)
    voronoi_gdf = voronoi_gdf.to_crs("EPSG:3857")
    interested_substations = interested_substations.to_crs("EPSG:3857")
    joined_gdf = gpd.sjoin_nearest(interested_substations, voronoi_gdf, how="left")
    interested_substations['cluster_label'] = joined_gdf['cluster_label']
    counts = interested_substations["cluster_label"].value_counts()
    interested_substations["hdbscan"] = interested_substations["cluster_label"].apply(lambda x: voronoi_gdf.loc[x, "demand"]/counts[x])


    corr_matrix.loc["GNN_hdbscan", location] = interested_substations["Demand (MVA)"].corr(interested_substations["hdbscan"])
    rmse_matrix.loc["GNN_hdbscan", location] = np.sqrt(((interested_substations["Demand (MVA)"] - interested_substations["hdbscan"]) ** 2).mean())
    mae_matrix.loc["GNN_hdbscan", location] = np.abs(interested_substations["Demand (MVA)"] - interested_substations["hdbscan"]).mean()

In [ ]:
print(len(interested_locations))

In [ ]:
train_count = 0
count = 0
results = {}
results["improvement"] = []
results["degradation"] = []
improvement_percentages = []
for i, location in enumerate(interested_locations):
    if rmse_matrix.loc["GNN", location] < rmse_matrix.loc["voronoi", location]:
        count += 1
        results["improvement"].append(location)

        percentage = (1 - (rmse_matrix.loc["GNN", location]/ rmse_matrix.loc["voronoi", location])) * 100
        improvement_percentages.append(percentage)

        if location in train_interested_locations.keys():
            train_count += 1
    else:
        results["degradation"].append(location)
print(f"Total {count} of {len(interested_locations)} locations have performance improvement in RMSE, {train_count}/{len(train_interested_locations.keys())} of them are in the train set.")
print(f"{results['improvement']} are improved.")
print(f"Maximum improvement: {max(improvement_percentages):.2f}%")
print(f"Minimum improvement: {min(improvement_percentages):.2f}%")
print(f"Average improvement: {np.mean(improvement_percentages):.2f}%")
print(f"{results['degradation']} are degraded.")

In [ ]:
# train_count = 0
# count = 0
# results = {}
# results["improvement"] = []
# results["degradation"] = []
# improvement_percentages = []
# for i, location in enumerate(interested_locations):
#     if mae_matrix.loc["GNN", location] < mae_matrix.loc["voronoi", location]:
#         count += 1
#         results["improvement"].append(location)
#
#         percentage = (1 - (mae_matrix.loc["GNN", location]/ mae_matrix.loc["voronoi", location])) * 100
#         improvement_percentages.append(percentage)
#
#         if location in train_interested_locations.keys():
#             train_count += 1
#     else:
#         results["degradation"].append(location)
# print(f"Total {count} of {len(interested_locations)} locations have performance improvement in MAE, {train_count}/{len(train_interested_locations.keys())} of them are in the train set.")
# print(f"Maximum improvement: {max(improvement_percentages):.2f}%")
# print(f"Minimum improvement: {min(improvement_percentages):.2f}%")
# print(f"Average improvement: {np.mean(improvement_percentages):.2f}%")
# print(f"{results['improvement']} are improved.")
# print(f"{results['degradation']} are degraded.")

In [ ]:
train_count = 0
count = 0
results = {}
results["improvement"] = []
results["degradation"] = []
improvement_percentages = []
for i, location in enumerate(interested_locations):
    if rmse_matrix.loc["GNN_hdbscan", location] < rmse_matrix.loc["civd_gpm", location]:
        count += 1
        results["improvement"].append(location)

        percentage = (1 - (rmse_matrix.loc["GNN_hdbscan", location]/ rmse_matrix.loc["civd_gpm", location])) * 100
        improvement_percentages.append(percentage)

        if location in train_interested_locations.keys():
            train_count += 1
    else:
        results["degradation"].append(location)
print(f"Total {count} of {len(interested_locations)} locations have performance improvement in RMSE by civd, {train_count}/{len(train_interested_locations.keys())} of them are in the train set.")
print(f"{results['improvement']} are improved.")
print(f"Maximum improvement: {max(improvement_percentages):.2f}%")
print(f"Minimum improvement: {min(improvement_percentages):.2f}%")
print(f"Average improvement: {np.mean(improvement_percentages):.2f}%")
print(f"{results['degradation']} are degraded.")

In [ ]:
# train_count = 0
# count = 0
# results = {}
# results["improvement"] = []
# results["degradation"] = []
# improvement_percentages = []
# for i, location in enumerate(interested_locations):
#     if mae_matrix.loc["GNN_hdbscan", location] < mae_matrix.loc["civd_gpm", location]:
#         count += 1
#         results["improvement"].append(location)
#
#         percentage = (1 - (mae_matrix.loc["GNN_hdbscan", location]/ mae_matrix.loc["civd_gpm", location])) * 100
#         improvement_percentages.append(percentage)
#
#         if location in train_interested_locations.keys():
#             train_count += 1
#     else:
#         results["degradation"].append(location)
# print(f"Total {count} of {len(interested_locations)} locations have performance improvement in MAE by civd, {train_count}/{len(train_interested_locations.keys())} of them are in the train set.")
# print(f"{results['improvement']} are improved.")
# print(f"Maximum improvement: {max(improvement_percentages):.2f}%")
# print(f"Minimum improvement: {min(improvement_percentages):.2f}%")
# print(f"Average improvement: {np.mean(improvement_percentages):.2f}%")
# print(f"{results['degradation']} are degraded.")

In [ ]:
rmse_matrix[list(train_interested_locations.keys())]

In [ ]:
rmse_matrix[list(test_interested_locations.keys())]

In [ ]:
mae_matrix[list(train_interested_locations.keys())]

In [ ]:
mae_matrix[list(test_interested_locations.keys())]